# CSA — kiểm cache thật và tạo từ vựng TRAIN (CPU)

**Accelerator None**, Internet ON. Add Input ba dataset:
`ducanh2403/visual-cache`, `ducanh2403/objectdetectionecache`, `vuthetam/mscoco-2014`.
Không cần checkpoint, prompt cache, UPT hay API token.

Đọc metadata/ID/filename của HDF5, không đọc tensor features; kiểm đủ123287ảnh,
detector records và12headerảnh lấy kích thước; tạo vocabulary từ train+restval.
Nếu đường mount khác, sửa bốn biến COCO_JSON, DETECTIONS, VISUAL_CACHE, IMAGE_ROOT ở cell đầu.
Giữ pinned hashes; khác payload thì gửi lỗi, không bỏ guard.

Chưa POS/CLIP text bank, chưa trích ROI hay train. Không dùng GPU. Worker cap30phút
là giới hạn dừng, không ETA. Save & Run All; gửi ZIP `csa_paper_asset_inventory_review.zip`.


In [ ]:
import hashlib, json, os, queue, signal, subprocess, sys, tempfile, threading, time, traceback, urllib.request, zipfile
from pathlib import Path
COMMIT = '9167917bd7cdf6a6284a332b4218923bcdb59d4b'
HASHES = {'audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'test_assets.py': '3bd78b47fb871fa828a0bde18307699a1041141980e34caa87ac94b7cb3ab1a4', 'IMPLEMENTATION_PROTOCOL.md': '0acd81ae858335c2d343f8a8a191806b0002b77acaa15aecdb8477eb2044f8fe'}
OUTPUT = Path('/kaggle/working/csa_paper_asset_inventory')
COCO_JSON = Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS = Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
VISUAL_CACHE = Path('/kaggle/input/datasets/ducanh2403/visual-cache')
IMAGE_ROOT = COCO_JSON.parent/'images'
if OUTPUT.exists(): raise RuntimeError('Preserve previous output/ZIP; use fresh runtime.')
OUTPUT.mkdir(parents=True)
SOURCE = Path(tempfile.mkdtemp(prefix='csa-assets-'))
LOG = OUTPUT/'notebook.log'
CONFIG = SOURCE/'config.json'
CONFIG.write_text(json.dumps(dict(dataset_json=str(COCO_JSON), detections=str(DETECTIONS),
    visual_cache=str(VISUAL_CACHE), image_root=str(IMAGE_ROOT), output=str(OUTPUT))), encoding='utf-8')
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=COMMIT, sha256=HASHES), indent=2), encoding='utf-8')
for name, expected in HASHES.items():
    payload = urllib.request.urlopen(f'https://raw.githubusercontent.com/Supzxjee/Image_Captioning/{COMMIT}/experiments/csa_paper/{name}', timeout=90).read()
    if hashlib.sha256(payload).hexdigest() != expected: raise RuntimeError('Source hash differs: '+name)
    (SOURCE/name).write_bytes(payload)
print('Pinned CPU audit source ready; GPU not required.', flush=True)


In [ ]:
def run(args, timeout=1800):
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Smoke active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Smoke wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete = False
try:
    run([sys.executable, '-u', SOURCE/'test_assets.py'], timeout=120)
    run([sys.executable, '-u', SOURCE/'audit_assets.py', '--config', CONFIG], timeout=1800)
    summary = json.loads((OUTPUT/'summary.json').read_text(encoding='utf-8'))
    complete = summary.get('status') == 'ASSET_INVENTORY_COMPLETE'
    if not complete: raise RuntimeError('Inventory incomplete')
    print(json.dumps(summary, indent=2), flush=True)
except BaseException:
    (OUTPUT/'notebook_failure.txt').write_text(traceback.format_exc(), encoding='utf-8')
    raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete, training_ready=False, model_executed=False)), encoding='utf-8')
    archive = OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file(): z.write(p, OUTPUT.name+'/'+str(p.relative_to(OUTPUT)))
    print('SEND:', archive, flush=True)
    print('INVENTORY COMPLETE; not training-ready.' if complete else 'INCOMPLETE: send ZIP/log.', flush=True)
